In [1]:
import yfinance as yf
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from torch import nn
import torch
from torch.utils.data import TensorDataset, DataLoader, WeightedRandomSampler
from xLSTM import sLSTM, mLSTM

from sklearn.metrics import roc_curve, auc

from pywt import wavedec, waverec


device=torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [2]:
def calculate_rsi(data, period=14):
    #calculate the relative strength index for an array like over [period] days
    #uses pandas
    delta = pd.DataFrame(data).diff()
    loss = delta.copy()
    gains = delta.copy()
    gains[gains < 0] = 0
    loss[loss > 0] = 0
    gain_wm = gains.rolling(period).apply(lambda x: np.sum(np.abs(x))/period)
    loss_wm = loss.rolling(period).apply(lambda x: np.sum(np.abs(x))/period)
    gpl = gain_wm/loss_wm
    return 100-100/(1+gpl)

# Download data, compute features

In [3]:
url = "https://raw.githubusercontent.com/datasets/s-and-p-500-companies/master/data/constituents.csv"
sp500 = pd.read_csv(url)

syms_array = sp500.sort_values('Date added')['Symbol'][:100].values
syms_list = syms_array.tolist()

In [4]:

#returns only
train_frac = 0.7
valtest_frac = 0.15


period = 14 #period for technical indicator calculation
prediction_window = 30 #how far ahead to predict directional price movement

train_frames = []
validate_frames = []
test_frames = []

for sym in syms_list:
    tick = yf.Ticker(sym)
    df = tick.history(period='20y', interval='1d')

    if df is None or df.empty:
        print('skipping', sym, 'no data found')
        continue

    if 'Close' not in df.columns or 'High' not in df.columns or 'Low' not in df.columns or 'Volume' not in df.columns:
        print('skipping', sym, 'no close found')
        continue

    close = df['Close']

    coeffs = wavedec(np.array(close), 'db4')

    coeffs[-1] = np.zeros_like(coeffs[-1])
    coeffs[-2] = np.zeros_like(coeffs[-2])
    coeffs[-3] = np.zeros_like(coeffs[-3])

    smooth = waverec(coeffs, 'db4')

    indicators = pd.DataFrame() #create the dataframe where the indicators will be calculated

    #daily returns
    indicators['ret'] = pd.DataFrame(smooth[:len(close)], index=close[:len(smooth)].index).pct_change()

    #make target columns
    indicators['target'] = close.pct_change(30)

    #add symbol column
    indicators['symbol'] = sym

    #indicators = indicators.replace([np.inf, -np.inf], np.nan)
    indicators_clean = indicators.dropna()

    train_length = int(len(indicators_clean)*train_frac)
    valtest_length = int(len(indicators_clean)*valtest_frac)



    train = indicators_clean[:train_length]
    validate = indicators_clean[train_length:train_length+valtest_length]
    test = indicators_clean[train_length+valtest_length:]

    

    if train.empty or validate.empty or test.empty:
        print('skipping', sym, 'no clean data')
        continue

    train_frames.append(train)
    validate_frames.append(validate)
    test_frames.append(test)

train_indicators = pd.concat(train_frames)
validate_indicators = pd.concat(validate_frames)
test_indicators = pd.concat(test_frames)

print('dataset size:', train_indicators.shape+validate_indicators.shape+test_indicators.shape)

$BF.B: possibly delisted; no price data found  (period=20y)


skipping BF.B no data found
dataset size: (339084, 3, 72588, 3, 72774, 3)


In [ ]:

#returns + technical indicators
train_frac = 0.7
valtest_frac = 0.15


period = 14 #period for technical indicator calculation
prediction_window = 30 #how far ahead to predict directional price movement

train_frames = []
validate_frames = []
test_frames = []

for sym in syms_list:
    tick = yf.Ticker(sym)
    df = tick.history(period='50y', interval='1d')

    if df is None or df.empty:
        print('skipping', sym, 'no data found')
        continue

    if 'Close' not in df.columns or 'High' not in df.columns or 'Low' not in df.columns or 'Volume' not in df.columns:
        print('skipping', sym, 'no close found')
        continue

    close = df['Close']
    high = df['High']
    low = df['Low']
    volume = df['Volume']

    coeffs = wavedec(np.array(close), 'db4')
    
    coeffs[-1] = np.zeros_like(coeffs[-1])
    coeffs[-2] = np.zeros_like(coeffs[-2])
    coeffs[-3] = np.zeros_like(coeffs[-3])

    smooth = waverec(coeffs, 'db4')

    smooth = pd.DataFrame(smooth[:len(close)], index=close[:len(smooth)].index).pct_change()
    
    indicators = pd.DataFrame() #create the dataframe where the indicators will be calculated

    #returns
    indicators['ret'] = smooth.pct_change()

    #stochastic K%
    indicators['stck'] = (smooth - low.rolling(period).min()) / (high.rolling(period).max() - low.rolling(period).min()) * 100

    #stochastic D% 
    indicators['stcd'] = indicators['stck'].rolling(3).sum()/3

    #momentum 
    indicators['mom'] = smooth.diff(period-1)

    #relative strength index
    indicators['rsi'] = calculate_rsi(smooth)

    #moving average convergence divergence
    emas = smooth.ewm(span = 12, adjust=False).mean() - smooth.ewm(span=26, adjust=False).mean()
    indicators['macd'] = emas

    #williams R%
    indicators['lwr'] = -100 * (high.rolling(period).max() - smooth) / (high.rolling(period).max() - low.rolling(period).min())

    #rate of change
    indicators['roc'] = 100 * smooth.diff(period)/smooth.shift(period)

    #Accumulation Distribution Oscillator
    indicators['ado'] = (high-smooth.shift(1))/(high-low) 

    #Commodity Channel Index
    tp = (smooth+high+low)/3
    stp = tp.rolling(period).sum() / period
    d = tp.rolling(period).apply(lambda x: np.mean(np.abs(x-x.mean())))
    indicators['cci'] = (tp-stp)/(0.015*d)

    #volume
    indicators['vol'] = volume

    #make target columns
    indicators['target'] = (np.sign(close.shift(-prediction_window)-close)>0).astype(int)

    #add symbol column
    indicators['symbol'] = sym

    indicators = indicators.replace([np.inf, -np.inf], np.nan)
    indicators_clean = indicators.dropna()

    train_length = int(len(indicators_clean)*train_frac)
    valtest_length = int(len(indicators_clean)*valtest_frac)

    indicators_clean['stck'] /= np.nanmax(indicators_clean['stck'][:train_length].abs())
    indicators_clean['stcd'] /= np.nanmax(indicators_clean['stcd'][:train_length].abs())
    indicators_clean['lwr'] /= np.nanmax(indicators_clean['lwr'][:train_length].abs())
    indicators_clean['macd'] /= np.nanmax(indicators_clean['macd'][:train_length].abs())
    indicators_clean['roc'] /= np.nanmax(indicators_clean['roc'][:train_length].abs())
    indicators_clean['rsi'] /= np.nanmax(indicators_clean['rsi'][:train_length].abs())
    indicators_clean['mom'] /= np.nanmax(indicators_clean['mom'][:train_length].abs())
    indicators_clean['ado'] /= np.nanmax(indicators_clean['ado'][:train_length].abs())
    indicators_clean['cci'] /= np.nanmax(indicators_clean['cci'][:train_length].abs())
    indicators_clean['vol'] /= np.nanmax(indicators_clean['vol'][:train_length].abs())


    train = indicators_clean[:train_length]
    validate = indicators_clean[train_length:train_length+valtest_length]
    test = indicators_clean[train_length+valtest_length:]

    

    if train.empty or validate.empty or test.empty:
        print('skipping', sym, 'no clean data')
        continue

    train_frames.append(train)
    validate_frames.append(validate)
    test_frames.append(test)

train_indicators = pd.concat(train_frames)
validate_indicators = pd.concat(validate_frames)
test_indicators = pd.concat(test_frames)

print('dataset size:', train_indicators.shape+validate_indicators.shape+test_indicators.shape)

In [ ]:
plt.imshow(train_indicators.drop(columns=['symbol','target']).corr()>0.9, cmap='coolwarm')
plt.xticks(ticks=np.arange(len(train_indicators.drop(columns=['symbol','target']).columns)), labels=train_indicators.drop(columns=['symbol', 'target']).columns)
plt.yticks(ticks=np.arange(len(train_indicators.drop(columns=['symbol','target']).columns)), labels=train_indicators.drop(columns=['symbol', 'target']).columns)
plt.colorbar()
plt.show()

# Create windows, train/val/test/split, pytorch data sets

In [5]:

if prediction_window < 30: 
    window_size=30
else:
    window_size = prediction_window

Xtrain_list=[]
ytrain_list=[]

Xvalidate_list=[]
yvalidate_list=[]

Xtest_list=[]
ytest_list=[]

for sym in syms_list:
    train = train_indicators[train_indicators['symbol']==sym].sort_index()
    validate = validate_indicators[validate_indicators['symbol']==sym].sort_index()
    test = test_indicators[test_indicators['symbol']==sym].sort_index()

    indicators = (train.drop(columns=['target', 'symbol']).values, 
                  validate.drop(columns=['target', 'symbol']).values, 
                  test.drop(columns=['target', 'symbol']).values)
    
    target = (train['target'].values, 
              validate['target'].values, 
              test['target'].values)
    
    for i in range(window_size,len(train[:-window_size])):
        Xtrain_list.append(indicators[0][i-window_size:i])
        ytrain_list.append(target[0][i-1])

    for i in range(window_size, len(validate[window_size:-window_size])):
        Xvalidate_list.append(indicators[1][i-window_size:i])
        yvalidate_list.append(target[1][i-1])

    for i in range(window_size, len(test[window_size:])):
        Xtest_list.append(indicators[2][i-window_size:i])
        ytest_list.append(target[2][i-1])


Xtrain=np.array(Xtrain_list,dtype=np.float32)
ytrain=np.array(ytrain_list,dtype=np.float32)
Xvalidate=np.array(Xvalidate_list,dtype=np.float32)
yvalidate=np.array(yvalidate_list,dtype=np.float32)
Xtest=np.array(Xtest_list,dtype=np.float32)
ytest=np.array(ytest_list,dtype=np.float32)

print(Xtrain.shape, ytrain.shape, Xvalidate.shape, yvalidate.shape, Xtest.shape, ytest.shape)

batch_size = 64
Xtrain_seq=Xtrain.reshape(-1, window_size, Xtrain.shape[2])
Xvalidate_seq=Xvalidate.reshape(-1, window_size, Xvalidate.shape[2])
Xtest_seq=Xtest.reshape(-1, window_size, Xtest.shape[2])




train_loader_seq=DataLoader(TensorDataset(torch.tensor(Xtrain_seq),torch.tensor(ytrain.reshape(-1,1))),batch_size=batch_size, drop_last=True)
val_loader_seq=DataLoader(TensorDataset(torch.tensor(Xvalidate_seq),torch.tensor(yvalidate.reshape(-1,1))),batch_size=batch_size, drop_last=True)
test_loader_seq=DataLoader(TensorDataset(torch.tensor(Xtest_seq),torch.tensor(ytest.reshape(-1,1))),batch_size=batch_size, drop_last=True)

(333144, 30, 1) (333144,) (63678, 30, 1) (63678,) (66834, 30, 1) (66834,)


# Define model and functions

In [6]:
class LSTMClassifier(nn.Module):
    def __init__(self, feature_dim, hidden_size=32, num_layers=1):
        super().__init__()
        #self.lstm=nn.LSTM(feature_dim,hidden_size,num_layers,batch_first=True)
        self.linin=nn.Linear(feature_dim, hidden_size)
        self.linout=nn.Linear(hidden_size,1)
        self.mlstm=mLSTM(hidden_size,hidden_size,num_layers,batch_first=True)
        self.slstm=sLSTM(hidden_size,hidden_size,num_layers,batch_first=True)
        self.sigmoid=nn.Sigmoid()
        #self.dropout=nn.Dropout(0.3)
        self.norm=nn.LayerNorm(hidden_size)
    def forward(self,x):
        out=self.linin(x.float())
        out,_=self.mlstm(out)
        out,_=self.slstm(out)
        out,_=self.mlstm(out)
        out,_=self.mlstm(out)
        out=self.norm(out)
        out=self.linout(out[:,-1,:])
        return out

def train_epoch(model,loader,crit,opt):
    model.train(); total=loss_sum=abs_err_sum=0
    for xb,yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        opt.zero_grad()
        pred=model(xb)
        loss=crit(pred,yb)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        opt.step()
        loss_sum+=loss.item()*len(xb)
        abs_err_sum += (pred - yb).abs().sum().item()
        total+=len(xb)
    return loss_sum/total, abs_err_sum/total

def eval_epoch(model,loader,crit):
    model.eval(); total=loss_sum=abs_err_sum=0; preds=[]; labels=[]
    with torch.no_grad():
        for xb,yb in loader:
            xb,yb=xb.to(device),yb.to(device)
            pred=model(xb)
            loss=crit(pred,yb)
            loss_sum+=loss.item()*len(xb)
            abs_err_sum += (pred - yb).abs().sum().item()            
            total+=len(xb)
            preds.extend(pred.cpu().numpy())
            labels.extend(yb.cpu().numpy().astype(int))
    return loss_sum/total, abs_err_sum/total, preds, labels

In [8]:
lr=1e-4
lstm=LSTMClassifier(feature_dim=Xtrain.shape[2]).to(device)
opt2=torch.optim.Adam(lstm.parameters(), lr=lr)
best_l=None; best_val_l=float("inf")
crit=nn.MSELoss()
print('prediction window:', window_size)


for epoch in range(1,11):
    tr_l,tr_a=train_epoch(lstm,train_loader_seq,crit,opt2)
    va_l,va_a,_,_=eval_epoch(lstm,val_loader_seq,crit)
    print(f"LSTM {epoch}: Train loss: {tr_l:.4f}, train mae: {tr_a:.4f}, Validation loss: {va_l:.4f}, validation mae: {va_a:.4f}")
    if va_l/best_val_l > 0.9:
        print('lowering lr')
        lr/=10
    if va_l<best_val_l: 
        best_val_l=va_l
        best_l=lstm.state_dict()

lstm.load_state_dict(best_l)

prediction window: 30
LSTM 1: Train loss: 0.0101, train mae: 0.0666, Validation loss: 0.0083, validation mae: 0.0692
LSTM 2: Train loss: 0.0019, train mae: 0.0269, Validation loss: 0.0008, validation mae: 0.0206
LSTM 3: Train loss: 0.0010, train mae: 0.0201, Validation loss: 0.0007, validation mae: 0.0194
LSTM 4: Train loss: 0.0009, train mae: 0.0187, Validation loss: 0.0007, validation mae: 0.0195
lowering lr
LSTM 5: Train loss: 0.0008, train mae: 0.0182, Validation loss: 0.0006, validation mae: 0.0186
lowering lr
LSTM 6: Train loss: 0.0008, train mae: 0.0180, Validation loss: 0.0006, validation mae: 0.0186
lowering lr
LSTM 7: Train loss: 0.0008, train mae: 0.0179, Validation loss: 0.0006, validation mae: 0.0184
lowering lr
LSTM 8: Train loss: 0.0007, train mae: 0.0177, Validation loss: 0.0006, validation mae: 0.0182
lowering lr
LSTM 9: Train loss: 0.0008, train mae: 0.0177, Validation loss: 0.0006, validation mae: 0.0182
lowering lr
LSTM 10: Train loss: 0.0007, train mae: 0.0176, Val

<All keys matched successfully>

In [186]:
(.0457-.0348)/.0457

0.23851203501094093

In [10]:
tl,ta,preds,labels=eval_epoch(lstm,test_loader_seq,crit)
print("LSTM Test Accuracy:", ta)

LSTM Test Accuracy: 0.01677444681232631


In [10]:
np.mean(np.abs(ytest))

np.float32(0.06937018)